In [ ]:
!pip install yfinance --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 116.6 MB/s eta 0:00:00
  Attempting uninstall: curl_cffi
    Found existing installation: curl_cffi 0.14.0
    Uninstalling curl_cffi-0.14.0:
      Successfully uninstalled curl_cffi-0.14.0
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.66
    Uninstalling yfinance-0.2.66:
      Successfully uninstalled yfinance-0.2.66


In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import time
import logging
#from yfinance import YFDataError # This line is removed

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Define the ticker for S&P 500 index
ticker = "^GSPC"

# Set the time period (last 30 days)
end_date = datetime.now()
start_date = end_date - timedelta(days=30)

# Initialize an empty DataFrame to store all data
all_data = pd.DataFrame()

# Download data in 7-day chunks with retry and exponential backoff
current_end = end_date
max_retries = 5
base_delay = 10  # seconds
chunk_delay = 5  # seconds between chunks

while current_end > start_date:
    current_start = max(current_end - timedelta(days=7), start_date)
    logger.info(f"Fetching data from {current_start} to {current_end}")

    for attempt in range(max_retries):
        try:
            # Download 1-minute data for the current chunk
            chunk = yf.download(ticker, start=current_start, end=current_end, interval="1m", auto_adjust=False)

            # Check if data is empty
            if not chunk.empty:
                # Append to the main DataFrame
                all_data = pd.concat([all_data, chunk])
                logger.info(f"Successfully fetched data for {current_start} to {current_end}")
                break  # Exit retry loop on success
            else:
                logger.warning(f"Empty data for {current_start} to {current_end}, retrying...")

        #except YFDataError as e: # This line is updated
        except Exception as e: # Catches any exception that now might occur during the download.
            logger.error(f"Error on attempt {attempt + 1}/{max_retries}: {e}")
            if attempt < max_retries - 1:
                # Exponential backoff: delay = base_delay * 2^attempt
                delay = base_delay * (2 ** attempt)
                logger.info(f"Waiting {delay} seconds before retrying...")
                time.sleep(delay)
                continue
            else:
                logger.error(f"Max retries reached for {current_start} to {current_end}, skipping...")

        # Delay between requests within a chunk
        time.sleep(2)

    # Delay between chunks
    time.sleep(chunk_delay)

    # Move the end date back
    current_end = current_start

# Remove any duplicates (in case of overlap)
all_data = all_data[~all_data.index.duplicated(keep='first')]

# Sort by datetime index
all_data = all_data.sort_index()

# Save to CSV
all_data.to_csv("AAPL.csv")
logger.info("Data saved to AAPL.csv")

# Display first few rows
logger.info("First few rows of data:")
print(all_data.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Price                        Adj Close        Close         High          Low  \
Ticker                           ^GSPC        ^GSPC        ^GSPC        ^GSPC   
Datetime                                                                        
2025-12-26 15:06:00+00:00  6932.560059  6932.560059  6933.169922  6931.479980   
2025-12-26 15:07:00+00:00  6932.680176  6932.680176  6933.180176  6932.180176   
2025-12-26 15:08:00+00:00  6931.080078  6931.080078  6932.750000  6930.990234   
2025-12-26 15:09:00+00:00  6932.439941  6932.439941  6932.439941  6931.149902   
2025-12-26 15:10:00+00:00  6930.859863  6930.859863  6932.459961  6930.410156   

Price                             Open   Volume  
Ticker                           ^GSPC    ^GSPC  
Datetime                                         
2025-12-26 15:06:00+00:00  6931.720215        0  
2025-12-26 15:07:00+00:00  6932.299805  3437855  
2025-12-26 15:08:00+00:00  6932.669922  4136454  
2025-12-26 15:09:00+00:00  6931.169922  3714153  
2

In [ ]:
all_data

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Datetime,,,,,,
2025-12-26 15:06:00+00:00,6932.560059,6932.560059,6933.169922,6931.479980,6931.720215,0
2025-12-26 15:07:00+00:00,6932.680176,6932.680176,6933.180176,6932.180176,6932.299805,3437855
2025-12-26 15:08:00+00:00,6931.080078,6931.080078,6932.750000,6930.990234,6932.669922,4136454
2025-12-26 15:09:00+00:00,6932.439941,6932.439941,6932.439941,6931.149902,6931.169922,3714153
2025-12-26 15:10:00+00:00,6930.859863,6930.859863,6932.459961,6930.410156,6932.399902,3872250
...,...,...,...,...,...,...
2026-01-23 20:55:00+00:00,6914.879883,6914.879883,6914.879883,6911.859863,6914.149902,26828000
2026-01-23 20:56:00+00:00,6914.250000,6914.250000,6915.109863,6913.390137,6914.890137,25209000


In [ ]:
# prompt: download all_data as csv in the computer

from google.colab import files
files.download('AAPL.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>